# Experiment 13 — JPEG compression sensitivity (NIH-only)

NIH radiographs are re-encoded at JPEG quality $q \in$ {50, 70, 90, 95},
re-embedded with RAD-DINO and DINOv3, and re-probed for exp02 (geometric S4),
exp03 (iso-blur $p$=4), and exp06 (reticular-fine-$p$=32). The RAD-DINO vs
DINOv3 CLS AUC dissociation is reported as a function of $q$; stability
across $q$ rules out the possibility that the MIMIC-CXR-JPG corpus's lossy
compression confounds the embedding-level results.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os, warnings
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators_work'))
QUALS = [50, 70, 90, 95]
out_dir = ROOT / 'v4_exp13_jpeg_sensitivity'
out_dir.mkdir(parents=True, exist_ok=True)

import sys; sys.path.insert(0, str(Path('..').resolve()))
try:
    from common.jpeg_sensitivity import run_jpeg_sweep
except Exception as e:
    warnings.warn(f'jpeg_sensitivity import failed: {e}')
    run_jpeg_sweep = None

if run_jpeg_sweep is None:
    pd.DataFrame({'status': ['module_absent']}).to_parquet(
        out_dir / 'exp13_manifest.parquet', index=False)
else:
    rows = run_jpeg_sweep(models=('raddino', 'dinov3'), qualities=QUALS,
        artifacts=('geometric_C1', 'geometric_S4', 'iso_blur_p4',
                   'reticular_fine_p32'),
        dataset='nih', out_dir=out_dir)
    print(f'jpeg sweep produced {len(rows)} rows')


/tmp/ipykernel_867557/2448806140.py:15: UserWarning: jpeg_sensitivity import failed: No module named 'common.jpeg_sensitivity'
  warnings.warn(f'jpeg_sensitivity import failed: {e}')
